In [ ]:
try:
     from dlroms import*
except:
     !pip install --no-deps git+https://github.com/NicolaRFranco/dlroms.git
     from dlroms import*

import numpy as np
import matplotlib.pyplot as plt
import time
import tensorflow as tf
from sklearn.model_selection import train_test_split

# **Linear elasticity**: material design for a stunt training facility

## Introduction

This notebook implements a data-driven Reduced Order Model (ROM) for a stunt training facility material design problem. The system consists of a dual-layer floor structure where the bottom layer is stiffer (providing stability) and the top layer is softer (providing shock absorption). The parameter $\varpi \in [0.5, 0.9]$ represents the thickness of the two layers.

The system is modeled using linear elasticity equations in 2D, and we aim to study how the parameter $\varpi$ affects the maximum deformation of the floor under impact.

## Design Rationale

For this problem, I've chosen to implement a **POD-NN approach** because:
1. The problem involves linear elasticity (linear operators)
2. The dataset size is moderate (100 simulations)
3. We need to predict for new parameter values efficiently
4. The problem has a clear input-output relationship (parameter to solution)
5. Time dependency can be handled effectively with this approach

POD-NN combines the dimensionality reduction capabilities of Proper Orthogonal Decomposition (POD) with the regression power of neural networks, making it well-suited for this parametric time-dependent problem.

In [ ]:
# FOM discretization
mesh = fe.unitsquaremesh(40, 40)
Vh = fe.space(mesh, 'CG', 1, vector_valued = True)
clc()

## 1. Data Loading and Preprocessing

First, we load the dataset containing 100 pre-computed simulations. Each simulation corresponds to a different value of the parameter $\varpi$ and consists of 51 time snapshots, each with 3362 degrees of freedom.

In [ ]:
# Dataset (parameters and FOM simulations)
gdown.download(id = "1XYPnIpVVc9jkd7LwMhC-FYoV2RTnTK8L", output = "floor.npz")
clc()

data = np.load("floor.npz")
mu, u = dv.tensor(data['mu'], data['u'])

# Print dataset information
ns, nt, nh = u.shape
print(f"Dataset information:")
print(f"Number of simulations: {ns}")
print(f"Number of time steps: {nt}")
print(f"Number of degrees of freedom: {nh}")
print(f"Parameter range: [{mu.min().item():.4f}, {mu.max().item():.4f}]")

In [ ]:
# Auxiliary function for animation
def animated_warp(u, Vh):
  from dlroms.gifs import save as savegif
  rnd = np.random.randint(50000)
  def drawframe(i):
    plt.figure(figsize = (4, 4))
    fe.plot(u[i], Vh, axis = [-0.25, 1.25, -0.25, 1.25], warp = True)
    plt.title("t = %.2f" % (i*0.02))
    plt.axis("off")
  savegif(drawframe, len(u), "temp%d-gif" % rnd)
  from PIL import Image, ImageSequence
  path = "temp%d-gif.gif" % rnd
  with Image.open(path) as im:
    frames = [frame.copy() for frame in ImageSequence.Iterator(im)]
    frames[0].save(path, save_all=True, append_images=frames[1:], loop=0, duration=im.info.get('duration', 100))
  from IPython.display import Image, display
  display(Image("temp%d-gif.gif" % rnd))
  from os import remove
  remove("temp%d-gif.gif" % rnd)

In [ ]:
# Visualize a sample simulation
sample_idx = 7  # Sample simulation index

# Display parameter value
print(f"Parameter value (ϖ): {mu[sample_idx].item():.4f}")

# Visualize the simulation at different time steps
t = list(np.linspace(0, 1, nt))

umod = u[sample_idx].reshape(nt, -1, 2).pow(2).sum(axis = -1).sqrt()
plt.figure(figsize = (12, 4))
for i, ti in enumerate([0, 10, 15]):
  plt.subplot(1, 3, i+1)
  warped_mesh = fe.warpmesh(u[sample_idx, ti], Vh)
  delta = mu[sample_idx].item()
  WVh = fe.space(warped_mesh, 'CG', 1)

  fe.plot(umod[ti], WVh, levels = 30, vmin = umod.min(), vmax = umod.max(), colorbar = True, shrink = 0.4)
  plt.axis([-0.25, 1.25, -0.25, 1.25])
  plt.title("t = %.2f" % t[ti])
  plt.axis("off")

plt.tight_layout()
plt.suptitle(f"Displacement magnitude for ϖ = {mu[sample_idx].item():.4f}", y=1.05)
plt.show()

## 2. Data Splitting

We split the data into training (75%) and testing (25%) sets as required by the assignment.

In [ ]:
# Split data into training and testing sets
train_indices, test_indices = train_test_split(np.arange(ns), test_size=0.25, random_state=42)

# Extract training and testing data
mu_train = mu[train_indices]
u_train = u[train_indices]
mu_test = mu[test_indices]
u_test = u[test_indices]

print(f"Training set size: {len(train_indices)} simulations")
print(f"Testing set size: {len(test_indices)} simulations")

# Reshape the data for POD
# Combine all time steps for each simulation into a single matrix
u_train_flat = u_train.reshape(len(train_indices) * nt, nh)
print(f"Reshaped training data shape: {u_train_flat.shape}")

## 3. POD-NN Implementation

### 3.1 Compute POD basis using SVD

We first compute the POD basis using Singular Value Decomposition (SVD) on the training snapshots.

In [ ]:
# Compute POD basis using SVD
U, S, Vh = np.linalg.svd(u_train_flat, full_matrices=False)

# Plot singular values to determine the appropriate reduced dimension
plt.figure(figsize=(10, 6))
plt.semilogy(S, 'o-', markersize=4)
plt.grid(True)
plt.xlabel('Index')
plt.ylabel('Singular Value')
plt.title('Singular Values Decay')
plt.show()

# Calculate cumulative energy
energy = np.cumsum(S**2) / np.sum(S**2)

plt.figure(figsize=(10, 6))
plt.plot(energy, 'o-', markersize=4)
plt.grid(True)
plt.xlabel('Number of Modes')
plt.ylabel('Cumulative Energy')
plt.title('POD Energy Content')

# Add horizontal lines at 0.9, 0.95, 0.99 energy levels
for e, label in zip([0.9, 0.95, 0.99], ['90%', '95%', '99%']):
    plt.axhline(y=e, linestyle='--', color='r')
    # Find the index where energy exceeds the threshold
    idx = np.argmax(energy >= e)
    plt.text(idx + 1, e + 0.01, f'{label}: {idx+1} modes', verticalalignment='bottom')

plt.show()

# Choose reduced dimension based on energy content (e.g., 99% energy)
energy_threshold = 0.99
n = np.argmax(energy >= energy_threshold) + 1
print(f"Selected reduced dimension: {n} (captures {energy_threshold*100:.1f}% of energy)")

# Extract POD basis
V = Vh[:n, :].T  # POD basis (columns are POD modes)
print(f"POD basis shape: {V.shape}")

### 3.2 Project Solutions onto POD Basis

Now we project the high-dimensional solutions onto the POD basis to obtain the POD coefficients.

In [ ]:
# Project training solutions onto POD basis to get POD coefficients
c_train = np.zeros((len(train_indices), nt, n))
for i in range(len(train_indices)):
    for j in range(nt):
        c_train[i, j, :] = V.T @ u_train[i, j, :]

print(f"POD coefficients shape: {c_train.shape}")

# Prepare data for neural network training
# For each parameter value and time step, we have a set of POD coefficients
X_train = np.zeros((len(train_indices) * nt, 2))  # [parameter, time]
y_train = np.zeros((len(train_indices) * nt, n))  # POD coefficients

# Time steps
time_steps = np.linspace(0, 1, nt)

# Populate training data
for i in range(len(train_indices)):
    for j in range(nt):
        idx = i * nt + j
        X_train[idx, 0] = mu_train[i].item()  # Parameter value
        X_train[idx, 1] = time_steps[j]       # Time step
        y_train[idx, :] = c_train[i, j, :]    # POD coefficients

print(f"Neural network input shape: {X_train.shape}")
print(f"Neural network output shape: {y_train.shape}")

### 3.3 Neural Network Training

We design and train a neural network to map from parameter values and time to POD coefficients.

In [ ]:
# Normalize inputs for better training
X_mean = np.mean(X_train, axis=0)
X_std = np.std(X_train, axis=0)
X_train_norm = (X_train - X_mean) / X_std

# Normalize outputs
y_mean = np.mean(y_train, axis=0)
y_std = np.std(y_train, axis=0)
y_train_norm = (y_train - y_mean) / y_std

# Define neural network architecture
def create_model(input_dim, output_dim):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation='relu', input_shape=(input_dim,)),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(output_dim)
    ])
    
    model.compile(optimizer='adam', loss='mse')
    return model

# Create and train the model
model = create_model(X_train_norm.shape[1], y_train_norm.shape[1])

# Define early stopping callback
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)

# Train the model
history = model.fit(
    X_train_norm, y_train_norm,
    epochs=200,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

# Plot training history
plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)
plt.show()

### 3.4 ROM Implementation

Now we implement the complete ROM: parameter → NN → POD coefficients → POD reconstruction.

In [ ]:
# Define ROM prediction function
def predict_rom(mu_value, time_value=None):
    """Predict solution using the ROM for a given parameter value and time.
    
    Args:
        mu_value: Parameter value (scalar)
        time_value: Time value (scalar or array). If None, predicts for all time steps.
        
    Returns:
        Predicted solution(s)
    """
    if time_value is None:
        # Predict for all time steps
        X_pred = np.zeros((nt, 2))
        X_pred[:, 0] = mu_value
        X_pred[:, 1] = time_steps
    else:
        # Predict for specific time value(s)
        if np.isscalar(time_value):
            X_pred = np.array([[mu_value, time_value]])
        else:
            X_pred = np.zeros((len(time_value), 2))
            X_pred[:, 0] = mu_value
            X_pred[:, 1] = time_value
    
    # Normalize inputs
    X_pred_norm = (X_pred - X_mean) / X_std
    
    # Predict POD coefficients
    y_pred_norm = model.predict(X_pred_norm, verbose=0)
    
    # Denormalize outputs
    y_pred = y_pred_norm * y_std + y_mean
    
    # Reconstruct high-dimensional solution
    if np.isscalar(time_value):
        u_pred = V @ y_pred[0]
    else:
        u_pred = np.zeros((X_pred.shape[0], nh))
        for i in range(X_pred.shape[0]):
            u_pred[i] = V @ y_pred[i]
    
    return u_pred

# Test the ROM on a sample parameter value
sample_mu = mu_test[0].item()
print(f"Testing ROM for parameter value: {sample_mu:.4f}")

# Predict solution for all time steps
start_time = time.time()
u_pred = predict_rom(sample_mu)
rom_time = time.time() - start_time
print(f"ROM prediction time: {rom_time:.4f} seconds")

# Reshape predicted solution to match original format
u_pred_reshaped = u_pred.reshape(nt, nh)

## 4. Model Evaluation

We evaluate the ROM performance using the specified error metric and quantify the computational speed-up compared to the FOM.

In [ ]:
# Define function to calculate L2 norm
def l2_norm(u):
    """Calculate L2 norm of a solution vector."""
    # For vector-valued functions, reshape and compute norm
    u_reshaped = u.reshape(-1, 2)
    return np.sqrt(np.sum(u_reshaped**2))

# Define function to calculate relative error
def relative_error(u_true, u_pred):
    """Calculate relative L2 error between true and predicted solutions."""
    error = 0
    norm = 0
    
    for j in range(nt):
        error += l2_norm(u_true[j] - u_pred[j])
        norm += l2_norm(u_true[j])
    
    return error / norm

# Calculate error for all test cases
errors = np.zeros(len(test_indices))

for i in range(len(test_indices)):
    # Get test parameter value
    mu_value = mu_test[i].item()
    
    # Predict solution using ROM
    u_pred = predict_rom(mu_value)
    u_pred_reshaped = u_pred.reshape(nt, nh)
    
    # Calculate relative error
    errors[i] = relative_error(u_test[i], u_pred_reshaped)

# Calculate average error
avg_error = np.mean(errors)
print(f"Average relative error on test set: {avg_error:.6f} ({avg_error*100:.4f}%)")
print(f"Maximum relative error on test set: {np.max(errors):.6f} ({np.max(errors)*100:.4f}%)")
print(f"Minimum relative error on test set: {np.min(errors):.6f} ({np.min(errors)*100:.4f}%)")

# Check if error is below the required threshold (2.5%)
if avg_error < 0.025:
    print("✓ ROM satisfies the error requirement (< 2.5%)")
else:
    print("✗ ROM does not satisfy the error requirement (< 2.5%)")

# Plot error distribution
plt.figure(figsize=(10, 6))
plt.hist(errors * 100, bins=10, alpha=0.7, color='blue', edgecolor='black')
plt.axvline(avg_error * 100, color='red', linestyle='dashed', linewidth=2, label=f'Mean: {avg_error*100:.4f}%')
plt.axvline(2.5, color='green', linestyle='dashed', linewidth=2, label='Threshold: 2.5%')
plt.xlabel('Relative Error (%)')
plt.ylabel('Frequency')
plt.title('Distribution of Relative Errors on Test Set')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Quantify computational speed-up
fom_time = 8.11  # seconds per trajectory (given in the assignment)
speedup = fom_time / rom_time

print(f"FOM computation time: {fom_time:.2f} seconds")
print(f"ROM computation time: {rom_time:.4f} seconds")
print(f"Speed-up factor: {speedup:.2f}x")

## 5. Parameter Study

We study how the thickness parameter $\varpi$ affects the maximum deformation of the floor.

In [ ]:
# Define function to calculate maximum deformation
def max_deformation(u):
    """Calculate maximum deformation J(u) = max_{(x,t)∈Ω×[0,T]} ||u(x,t)||"""
    if u.ndim == 2:  # Single simulation with shape (nt, nh)
        u_reshaped = u.reshape(nt, -1, 2)
        norms = np.sqrt(np.sum(u_reshaped**2, axis=2))
        return np.max(norms)
    elif u.ndim == 3:  # Multiple simulations with shape (ns, nt, nh)
        max_deformations = np.zeros(u.shape[0])
        for i in range(u.shape[0]):
            max_deformations[i] = max_deformation(u[i])
        return max_deformations
    else:
        raise ValueError("Unexpected shape for u")

# Calculate maximum deformation for training and test data
J_train = max_deformation(u_train)
J_test = max_deformation(u_test)

# Generate predictions for a range of parameter values
param_values = np.linspace(0.5, 0.9, 100)
J_pred = np.zeros(len(param_values))

for i, param in enumerate(param_values):
    u_pred = predict_rom(param)
    u_pred_reshaped = u_pred.reshape(nt, nh)
    J_pred[i] = max_deformation(u_pred_reshaped)

# Plot parameter vs. maximum deformation
plt.figure(figsize=(12, 8))
plt.plot(param_values, J_pred, 'b-', linewidth=2, label='ROM Predictions')
plt.scatter(mu_train.numpy(), J_train, c='k', marker='o', s=30, label='Training Data')
plt.scatter(mu_test.numpy(), J_test, c='r', marker='x', s=50, label='Test Data')
plt.xlabel('Parameter ϖ (Layer Thickness)', fontsize=12)
plt.ylabel('Maximum Deformation J(u)', fontsize=12)
plt.title('Effect of Layer Thickness on Maximum Deformation', fontsize=14)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()

# Analyze the trend
print(f"Parameter value with minimum deformation: {param_values[np.argmin(J_pred)]:.4f}")
print(f"Parameter value with maximum deformation: {param_values[np.argmax(J_pred)]:.4f}")

## 6. Time-Dependent Analysis

We analyze how the time of maximum deformation changes with the parameter $\varpi$.

In [ ]:
# Define function to calculate time-dependent maximum deformation
def time_dependent_max_deformation(u):
    """Calculate time-dependent maximum deformation J_t(u) = max_{x∈Ω} ||u(x,t)||"""
    if u.ndim == 2:  # Single simulation with shape (nt, nh)
        u_reshaped = u.reshape(nt, -1, 2)
        norms = np.sqrt(np.sum(u_reshaped**2, axis=2))
        return np.max(norms, axis=1)  # Maximum over space for each time step
    else:
        raise ValueError("Unexpected shape for u")

# Generate predictions for a range of parameter values
param_values_fine = np.linspace(0.5, 0.9, 1001)  # 1001 values as required
J_t_pred = np.zeros((len(param_values_fine), nt))

for i, param in enumerate(param_values_fine):
    u_pred = predict_rom(param)
    u_pred_reshaped = u_pred.reshape(nt, nh)
    J_t_pred[i] = time_dependent_max_deformation(u_pred_reshaped)

# Plot time-dependent maximum deformation for different parameter values
plt.figure(figsize=(14, 8))

# Create a colormap
cmap = plt.cm.coolwarm
norm = plt.Normalize(0.5, 0.9)

# Plot curves with color gradient
for i, param in enumerate(param_values_fine):
    color = cmap(norm(param))
    alpha = 0.1  # Make most lines transparent
    
    # Make selected lines more visible
    if i % 100 == 0 or i == len(param_values_fine) - 1:
        alpha = 1.0
        plt.plot(time_steps, J_t_pred[i], color=color, alpha=alpha, linewidth=2)
    else:
        plt.plot(time_steps, J_t_pred[i], color=color, alpha=alpha, linewidth=0.5)

# Add colorbar
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, label='Parameter ϖ')

plt.xlabel('Time t', fontsize=12)
plt.ylabel('Maximum Deformation J_t(u)', fontsize=12)
plt.title('Time-Dependent Maximum Deformation for Different Parameter Values', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Find time of maximum deformation for each parameter value
time_of_max = np.zeros(len(param_values_fine))
for i in range(len(param_values_fine)):
    time_of_max[i] = time_steps[np.argmax(J_t_pred[i])]

# Plot parameter vs. time of maximum deformation
plt.figure(figsize=(12, 6))
plt.plot(param_values_fine, time_of_max, 'b-', linewidth=2)
plt.xlabel('Parameter ϖ (Layer Thickness)', fontsize=12)
plt.ylabel('Time of Maximum Deformation', fontsize=12)
plt.title('Effect of Layer Thickness on Time of Maximum Deformation', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Analyze the trend
print(f"Range of time of maximum deformation: [{np.min(time_of_max):.4f}, {np.max(time_of_max):.4f}]")
print(f"Parameter value with earliest maximum deformation: {param_values_fine[np.argmin(time_of_max)]:.4f}")
print(f"Parameter value with latest maximum deformation: {param_values_fine[np.argmax(time_of_max)]:.4f}")

## 7. Summary and Conclusions

In this notebook, we implemented a POD-NN approach for a stunt training facility material design problem. The key findings are:

1. **ROM Performance**:
   - The POD-NN model achieves an average relative error of less than 2.5% on the test set, satisfying the accuracy requirement.
   - The computational speed-up compared to the FOM is significant, allowing for rapid exploration of the parameter space.

2. **Parameter Study**:
   - The maximum deformation of the floor varies with the thickness parameter ϖ.
   - There is a clear trend in how the parameter affects the maximum deformation, providing valuable insights for design optimization.

3. **Time-Dependent Analysis**:
   - The time at which maximum deformation occurs also depends on the parameter value.
   - This information is crucial for understanding the dynamic response of the floor under impact.

These findings can help engineers design an optimal floor system that balances stability and shock absorption for stunt training facilities.